# AI3.3 — Qwen3 0.6B LoRA

This notebook reuses `/kaggle/working/ai3.3` when a complete project copy is already there. It never clones over an existing folder and does not pull updates automatically. It clones from GitHub only when that folder is absent. The notebook runs a **dry run by default**; `START_TRAINING` stays `False` until you deliberately change it. Replit does not control Kaggle sessions.

For a private repository, use an authorized GitHub/Kaggle connection or Kaggle's secure secret storage. Never paste credentials into cells, URLs, or notebook output.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/alvinhuang114-cloud/ai3.3.git"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/ai3.3")
UPDATE_EXISTING_REPO = False
# Set True only when intentionally updating a clean Git checkout.
REQUIRED_FILES = (
    Path("training/requirements-kaggle.txt"),
    Path("training/configs/qwen3_0_6b_lora.json"),
    Path("training/src/ai33_training/train.py"),
    Path("training/examples/sample_chat.jsonl"),
)

def validate_project(path):
    missing = [
        str(relative)
        for relative in REQUIRED_FILES
        if not (path / relative).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            f"Project folder is incomplete: {path}. Missing: {', '.join(missing)}. "
            "No clone was attempted. Correct REPO_DIR or update the existing copy."
        )

if REPO_DIR.exists():
    validate_project(REPO_DIR)
    if UPDATE_EXISTING_REPO:
        if not (REPO_DIR / ".git").exists():
            raise RuntimeError(
                "UPDATE_EXISTING_REPO requires a Git checkout; no update was attempted."
            )
        working_changes = subprocess.run(
            ["git", "-C", str(REPO_DIR), "status", "--porcelain"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()
        if working_changes:
            raise RuntimeError(
                "The existing Git checkout has local changes. Save them before updating."
            )
        origin = subprocess.run(
            ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().rstrip("/")
        expected_origin = REPO_URL.rstrip("/")
        if origin.removesuffix(".git") != expected_origin.removesuffix(".git"):
            raise RuntimeError(
                "The Git checkout does not use the expected AI3.3 remote; no update was attempted."
            )
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )
        validate_project(REPO_DIR)
        print("Updated the existing AI3.3 checkout with a fast-forward-only pull.")
    else:
        print("Using the existing AI3.3 folder; no clone or pull was run.")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
    validate_project(REPO_DIR)
    print("Cloned the AI3.3 repository:", REPO_DIR)

print("Repository is ready:", REPO_DIR)

In [ ]:
# Kaggle supplies its own CUDA-enabled PyTorch; this requirements file leaves torch untouched.
requirements = REPO_DIR / "training" / "requirements-kaggle.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)

## Set your dataset and run mode

Attach your dataset through Kaggle's **Add Input** control. Leave `TRAIN_FILE` empty to validate the bundled sample, or set it to your JSONL path to validate your dataset. Keep `START_TRAINING = False` for the preflight check. It checks the model configuration and tokenizer but does not load model weights or train; it needs internet access for the small Hugging Face config/tokenizer files.

In [ ]:
TRAIN_FILE = ""  # Example: /kaggle/input/my-dataset/train.jsonl
EVAL_FILE = ""   # Optional separate JSONL validation file
START_TRAINING = False

training_dir = REPO_DIR / "training"
command = [
    sys.executable,
    "-m",
    "ai33_training.train",
    "--config",
    "configs/qwen3_0_6b_lora.json",
]

if TRAIN_FILE:
    command.extend(["--train-file", TRAIN_FILE])
if EVAL_FILE:
    command.extend(["--eval-file", EVAL_FILE])

if not TRAIN_FILE:
    print("TRAIN_FILE is empty; the bundled sample dataset will be used for the dry run.")

if START_TRAINING:
    if not TRAIN_FILE:
        raise ValueError("Set TRAIN_FILE to your attached dataset before training.")
    command.append("--run-training")
else:
    command.append("--dry-run")

environment = os.environ.copy()
source_dir = str(training_dir / "src")
environment["PYTHONPATH"] = source_dir + os.pathsep + environment.get("PYTHONPATH", "")
subprocess.run(command, cwd=training_dir, env=environment, check=True)

## Evaluate a saved adapter (optional)

After training completes, set `ADAPTER_DIR` to the adapter directory and `EVAL_FILE` to a validation JSONL file, then run this cell. Evaluation also requires a Kaggle GPU.

In [ ]:
ADAPTER_DIR = ""  # Example: /kaggle/working/ai3.3/training/outputs/qwen3-0.6b-lora/adapter
if ADAPTER_DIR:
    if not EVAL_FILE:
        raise ValueError("Set EVAL_FILE before running standalone evaluation.")
    eval_command = [
        sys.executable,
        "-m",
        "ai33_training.evaluate",
        "--config",
        "configs/qwen3_0_6b_lora.json",
        "--checkpoint",
        ADAPTER_DIR,
        "--eval-file",
        EVAL_FILE,
    ]
    subprocess.run(eval_command, cwd=training_dir, env=environment, check=True)
else:
    print("Evaluation skipped. Set ADAPTER_DIR and EVAL_FILE when ready.")